# Part 1: Data collection and preparation

Owner: _(name)_

**What this notebook is for.** Check that the data file is clean and complete, and write down where it
came from. Everyone else relies on this check.

**What you do.** Run the cells from top to bottom (Shift+Enter). Where a cell says *write here*, replace
the grey example text with your own sentences. Those sentences become section 1 of the report
(brief, section 1: source, period, order, duplicates, missing values, closures versus gaps, outliers,
transformations).

**What you hand over.** This notebook with its outputs, and your text for the report's data section.

You never change `data/sp500_daily.csv` here. If something looks wrong with it, say so in the group.

### Setup: run this cell first (just run, nothing to change)

It fetches the repository (in Colab), loads the S&P 500 data and the shared settings, and prints the
three periods: train, validation, test. The first run in Colab may take a minute while it installs a
package or two.

If Colab asks for a GitHub token every time, add the token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar). Step-by-step in `docs/START_HERE.md`.

In [ ]:
# ---- Setup (identical in every notebook). Just run it. ----
import sys, os, pathlib, subprocess, importlib.util

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"
EXTRA_PACKAGES = ['pandas_market_calendars']      # what this notebook needs that Colab does not have by default

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. In Colab, fetch the repository (once per session). On your own computer this step is skipped.
if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")             # the Colab secret, see docs/START_HERE.md
    except Exception:
        from getpass import getpass
        token = getpass("GitHub token (press Enter if the repository is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    os.chdir("sp500-forecasting")

# 2. Install anything missing (usually only on the first run in Colab).
missing = [p for p in EXTRA_PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("installing", ", ".join(missing), "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# 3. Find the repository folder and load the shared code (src/config.py and src/common.py).
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

# 4. The data everyone uses, and the three periods in time order.
df = common.load_prices()           # columns: Open, High, Low, Close, Volume, log_return; one row per trading day
y = common.target_series(df)        # the daily log return: what the models work on
train, val, test = common.split(y)  # dates from src/config.py
print("repository:", root)
print("data:", df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train       {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)  the model learns here")
print(f"validation  {val.index.min().date()} to {val.index.max().date()}  ({len(val)} days)  you compare settings here")
print(f"test        {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)  touched once, at the very end")
df.tail(3)

## 1. Where the data came from (just run)

The download script wrote this record when it fetched the data: source, series identifier, access date,
period and sample size. The brief asks for all five.

In [ ]:
import json
meta = json.loads((root / "data" / "sp500_daily.meta.json").read_text())
for key, value in meta.items():
    print(f"{key:24s} {value}")

**Write here: why this period?** For example: it gives the models enough history to learn from; it is
recent enough to matter; it contains calm years and stressed ones (2020); it starts after the 2008 crisis
so that one extreme regime does not dominate.

_your text_

## 2. First look at the file (just run)

`info()` lists the columns and how many rows have a value in each. `describe()` shows the count, mean,
spread (std), smallest and largest value per column.

In [ ]:
df.info()
df[["Open", "High", "Low", "Close", "Volume"]].describe().round(2)

## 3. Is the file in time order, with no repeated dates and no empty cells? (just run)

What you want to see: `True`, then `0`, then a `0` for every column (the first `log_return` is empty by
construction, because there is no "yesterday" for the first day).

In [ ]:
print("rows are in time order:", df.index.is_monotonic_increasing)
print("repeated dates:", int(df.index.duplicated().sum()))
print("empty cells per column:")
print(df.isna().sum())

## 4. Missing days: a closed market is not a gap (just run)

The stock market is closed on weekends and on holidays, so those dates are not in the file, and that is
correct. A real gap would be a trading day with no row. The cell compares the dates in the file with
the New York Stock Exchange calendar. What you want to see: `0` real gaps and `0` rows on non-trading days.

In [ ]:
try:
    import pandas_market_calendars as pmc
    nyse = pmc.get_calendar("NYSE")
    sched = nyse.schedule(start_date=df.index.min(), end_date=df.index.max())
    trading_days = pd.DatetimeIndex(sched.index.date)
    missing = trading_days.difference(df.index.normalize())     # calendar says open, file has no row
    extra = df.index.normalize().difference(trading_days)       # file has a row, calendar says closed
    print("trading days in the calendar:", len(trading_days), "| rows in the file:", len(df))
    print("trading days with no row (real gaps):", len(missing)); print(missing[:20])
    print("rows on non-trading days:", len(extra)); print(extra[:20])
except ImportError:
    print("pandas_market_calendars is not installed; showing the longest gaps between rows instead (3 days = a normal weekend)")
    gaps = df.index.to_series().diff().dt.days
    print(gaps.sort_values(ascending=False).head(10))

## 5. Extreme days, also called outliers (just run)

For a price series an outlier is a day with an unusually big *move*, not an unusually high *price*.
The cell lists the days whose daily return was more than 4 standard deviations away from the average
(a standard deviation is the usual size of a daily move). These are real events, for example March 2020,
not typing errors. We keep them, and the report names them.

In [ ]:
r = df["log_return"].dropna()
z = (r - r.mean()) / r.std()           # how many standard deviations away from the average each day is
extreme = r[z.abs() > 4]
print(len(extreme), "days beyond 4 standard deviations")
print("biggest falls:"); print(extreme.sort_values().head(10))
print("biggest rises:"); print(extreme.sort_values().tail(10))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
df["Close"].plot(ax=ax[0], title="S&P 500 daily close (index points)")
r.plot(ax=ax[1], title="daily log return (about the percentage change per day)")
plt.tight_layout()

## 6. Your decisions (write here)

One line each, with the reason. Replace the examples with what you actually found above.

- Missing trading days: none found, so nothing to fill in or drop.
- Repeated dates: none.
- Extreme days: kept, because they are real market events the models must face.
- Transformation: the models work on the daily log return (set in `src/config.py`), not on the price,
  because the return stays around a fixed level while the price drifts upward. Every forecast is turned
  back into a closing value in index points before it is scored (`common.to_price` does this).
- Scaling: the neural networks squeeze their inputs into a fixed range using only the training period's
  smallest and largest values, so nothing from later periods leaks in.

## 7. The three periods (just run)

These dates come from `src/config.py`. Part 2 confirms them. Copy this table into the report.

In [ ]:
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:11s} {part.index.min().date()} to {part.index.max().date()}  {len(part):5d} days")